# Step 9 — Model Training & Hyperparameter Tuning
### Credit Risk Prediction — Lending Club Dataset

Continues from the completed **Step 7** (leakage-safe preprocessing +
train/test split) and **Step 8** (leakage-safe PCA + SMOTE pipeline
*structure*, diagnostic-only).

**Goal:** train and tune two models with proper cross-validation.

- **Model A:** `preprocessing → StandardScaler → PCA(0.95) → SMOTE → LogisticRegression`
- **Model B:** `preprocessing → SMOTE → DecisionTreeClassifier`

**Methodology:**
- Fits use the **original** `X_train` / `y_train` — **not** the diagnostic
  `fit_resample()` arrays produced in Step 8. Those arrays only existed to
  report component counts / class balances and are never touched here.
- Both pipelines are `imblearn.pipeline.Pipeline`, so `PCA`/`StandardScaler`/
  `SMOTE` are re-fit **inside every training fold** of cross-validation —
  each validation fold is scored on data SMOTE never saw.
- `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` is used for
  both grid searches.
- `X_test` is not loaded into any estimator or scorer in this notebook.

**This section does NOT:**
- evaluate anything on `X_test`
- perform threshold tuning
- add Random Forest / XGBoost
- run SHAP / any explainability step


In [ ]:
# Ensure imbalanced-learn is available (Colab does not ship it by default)
try:
    import imblearn
except ImportError:
    %pip install -q imbalanced-learn
    import imblearn

print("imbalanced-learn version:", imblearn.__version__)

imbalanced-learn version: 0.14.2


## 1. Mount Google Drive & retrieve Step 7 artifacts

This is a separate notebook from Steps 7–8, so it retrieves everything it
needs from Drive rather than assuming any variables are already in memory.

Reloads the Step 6 CSV from `PROJECT_DIR` and reproduces the exact Step 7
split and preprocessing definition (`random_state=42`, same feature lists),
so the resulting `X_train` / `X_test` / `preprocessor` are identical to the
originals. **Only `X_train`/`y_train` are used for fitting below —
`X_test`/`y_test` are loaded for completeness but not touched otherwise.**

In [ ]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Drive already mounted.")

PROJECT_DIR = '/content/drive/MyDrive/Credit_Risk_LendingClub'
os.makedirs(PROJECT_DIR, exist_ok=True)
print("PROJECT_DIR:", PROJECT_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/Credit_Risk_LendingClub


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', 100)

DATA_PATH = "credit_risk_step6_final.csv"  # already loaded in the Colab session

df = pd.read_csv(DATA_PATH, low_memory=False)
df['issue_d'] = pd.to_datetime(df['issue_d'])
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'])
assert df.shape == (41988, 37), f"Unexpected shape: {df.shape}"
print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")

RANDOM_STATE = 42
train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df['target'], random_state=RANDOM_STATE
)

NON_FEATURE_COLS = ['target', 'issue_d', 'earliest_cr_line']
X_train = train_df.drop(columns=NON_FEATURE_COLS)
y_train = train_df['target'].copy()
X_test = test_df.drop(columns=NON_FEATURE_COLS)
y_test = test_df['target'].copy()

numerical_features = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'annual_inc', 'dti',
    'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
    'open_acc', 'revol_bal', 'revol_util', 'total_acc', 'delinq_amnt',
    'pub_rec_bankruptcies', 'credit_history_months', 'loan_to_income',
    'installment_to_income', 'revol_bal_to_income', 'open_acc_ratio',
    'fico_avg', 'grade_ordinal', 'emp_length_years',
    'has_public_record', 'has_delinquency', 'pub_rec_bankruptcies_missing',
    'revol_util_missing', 'emp_length_missing', 'is_income_verified'
]
categorical_features = [
    'verification_status', 'addr_state', 'dti_bin',
    'home_ownership_grouped', 'purpose_grouped'
]
assert set(numerical_features) | set(categorical_features) == set(X_train.columns)

numeric_pipe = Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])
categorical_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipe, numerical_features),
    ('cat', categorical_pipe, categorical_features),
])
preprocessor.fit(X_train)

print()
print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape: ", y_test.shape)
print()
print("Class distribution in y_train (counts):")
print(y_train.value_counts())

Loaded dataset: 41,988 rows x 37 columns

X_train shape: (33590, 34)
X_test shape:  (8398, 34)
y_train shape: (33590,)
y_test shape:  (8398,)

Class distribution in y_train (counts):
target
0    28458
1     5132
Name: count, dtype: int64


## 2. Imports, cross-validation strategy & scoring metric

### Why `average_precision` (PR-AUC) as the scoring metric

The target is imbalanced (~85% good loans / ~15% bad loans, confirmed
above). Two candidate metrics were considered:

- **`roc_auc`** — measures ranking quality against the true-negative rate,
  but with an 85/15 split the true-negative count is large, so ROC-AUC can
  stay high even when the model is poor at ranking the minority ("bad
  loan") class specifically. It tends to look optimistic under imbalance.
- **`average_precision`** (area under the precision-recall curve) —
  evaluates precision and recall **with respect to the positive (minority,
  "bad loan") class only**, with no true-negative term to inflate the
  score. This makes it far more sensitive to how well the model actually
  identifies defaults, which is the business-relevant class here (missing
  a bad loan is costlier than the reverse).

`average_precision` is therefore used as the `scoring` metric in both
`GridSearchCV` runs below. `cv.get_n_splits()` and the CV object are shared
by both grids so the two models are tuned under identical conditions.

In [ ]:
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
SCORING_METRIC = 'average_precision'  # PR-AUC -- see justification above

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"CV strategy: {cv}")
print(f"Number of folds: {cv.get_n_splits()}")
print(f"Scoring metric: {SCORING_METRIC}")

CV strategy: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
Number of folds: 5
Scoring metric: average_precision


## 3. Model A — Logistic Regression

Pipeline: `preprocessing → StandardScaler → PCA(n_components=0.95) → SMOTE → LogisticRegression`

`clone(preprocessor)` gives this pipeline its own unfitted copy of the
Step 7 `ColumnTransformer` definition, so it is refit inside every CV fold
rather than reusing the object already fitted on the full `X_train` above.

**Grid** — a small, sensible set of `(penalty, solver)` combinations that
are actually compatible in scikit-learn, each swept over a handful of `C`
values:
- `l2` + `lbfgs` (default combo)
- `l1` + `liblinear`
- `l2` + `liblinear`

In [ ]:
pipeline_lr = ImbPipeline(steps=[
    ('preprocessing', clone(preprocessor)),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

param_grid_lr = [
    {'classifier__penalty': ['l2'], 'classifier__solver': ['lbfgs'],
     'classifier__C': [0.01, 0.1, 1, 10]},
    {'classifier__penalty': ['l1'], 'classifier__solver': ['liblinear'],
     'classifier__C': [0.01, 0.1, 1, 10]},
    {'classifier__penalty': ['l2'], 'classifier__solver': ['liblinear'],
     'classifier__C': [0.01, 0.1, 1, 10]},
]

print("Pipeline A structure:")
for name, step in pipeline_lr.steps:
    print(f"  {name:12s} -> {step}")
print()
print(f"Grid size: {sum(len(d['classifier__C']) for d in param_grid_lr)} "
      f"parameter combinations x {cv.get_n_splits()} folds = "
      f"{sum(len(d['classifier__C']) for d in param_grid_lr) * cv.get_n_splits()} fits")

Pipeline A structure:
  preprocessing -> ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['loan_amnt', 'term', 'int_rate',
                                  'installment', 'annual_inc', 'dti',
                                  'inq_last_6mths', 'mths_since_last_delinq',
                                  'mths_since_last_record', 'open_acc',
                                  'revol_bal', 'revol_util', 'total_acc',
                                  'delinq_amnt', 'pub_rec_bankruptcies',
                                  'credit_history_months', 'loan_to...
                                  'has_delinquency',
                                  'pub_rec_bankruptcies_missing',
                                  'revol_util_missing', 'emp_length_missing',
                                  'is_income_verified']),
           

In [ ]:
grid_lr = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=param_grid_lr,
    scoring=SCORING_METRIC,
    cv=cv,
    n_jobs=-1,
    refit=True,
)

grid_lr.fit(X_train, y_train)  # X_test is never passed in here

print("=" * 60)
print("MODEL A -- Logistic Regression -- GridSearchCV results")
print("=" * 60)
print("Best parameters:       ", grid_lr.best_params_)
print(f"Best CV score ({SCORING_METRIC}): {grid_lr.best_score_:.4f}")
print("Number of CV folds:    ", cv.get_n_splits())
print()
print("Best estimator pipeline structure:")
for name, step in grid_lr.best_estimator_.steps:
    print(f"  {name:12s} -> {step}")

MODEL A -- Logistic Regression -- GridSearchCV results
Best parameters:        {'classifier__C': 10, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}
Best CV score (average_precision): 0.2931
Number of CV folds:     5

Best estimator pipeline structure:
  preprocessing -> ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['loan_amnt', 'term', 'int_rate',
                                  'installment', 'annual_inc', 'dti',
                                  'inq_last_6mths', 'mths_since_last_delinq',
                                  'mths_since_last_record', 'open_acc',
                                  'revol_bal', 'revol_util', 'total_acc',
                                  'delinq_amnt', 'pub_rec_bankruptcies',
                                  'credit_history_months', 'loan_to...
                   

## 4. Model B — Decision Tree

Pipeline: `preprocessing → SMOTE → DecisionTreeClassifier`

No scaling or PCA — tree splits are invariant to monotonic feature scaling,
and keeping the native feature space preserves interpretability for a
Decision Tree.

**Grid** — a small, sensible sweep over the four requested hyperparameters.

In [ ]:
pipeline_dt = ImbPipeline(steps=[
    ('preprocessing', clone(preprocessor)),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('classifier', DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

param_grid_dt = {
    'classifier__criterion': ['gini', 'entropy'],
    'classifier__max_depth': [4, 8, 12, None],
    'classifier__min_samples_split': [2, 10],
    'classifier__min_samples_leaf': [1, 5],
}

print("Pipeline B structure:")
for name, step in pipeline_dt.steps:
    print(f"  {name:12s} -> {step}")
print()
_n_combos = (len(param_grid_dt['classifier__criterion'])
             * len(param_grid_dt['classifier__max_depth'])
             * len(param_grid_dt['classifier__min_samples_split'])
             * len(param_grid_dt['classifier__min_samples_leaf']))
print(f"Grid size: {_n_combos} parameter combinations x {cv.get_n_splits()} folds "
      f"= {_n_combos * cv.get_n_splits()} fits")

Pipeline B structure:
  preprocessing -> ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['loan_amnt', 'term', 'int_rate',
                                  'installment', 'annual_inc', 'dti',
                                  'inq_last_6mths', 'mths_since_last_delinq',
                                  'mths_since_last_record', 'open_acc',
                                  'revol_bal', 'revol_util', 'total_acc',
                                  'delinq_amnt', 'pub_rec_bankruptcies',
                                  'credit_history_months', 'loan_to...
                                  'has_delinquency',
                                  'pub_rec_bankruptcies_missing',
                                  'revol_util_missing', 'emp_length_missing',
                                  'is_income_verified']),
           

In [ ]:
grid_dt = GridSearchCV(
    estimator=pipeline_dt,
    param_grid=param_grid_dt,
    scoring=SCORING_METRIC,
    cv=cv,
    n_jobs=-1,
    refit=True,
)

grid_dt.fit(X_train, y_train)  # X_test is never passed in here

print("=" * 60)
print("MODEL B -- Decision Tree -- GridSearchCV results")
print("=" * 60)
print("Best parameters:       ", grid_dt.best_params_)
print(f"Best CV score ({SCORING_METRIC}): {grid_dt.best_score_:.4f}")
print("Number of CV folds:    ", cv.get_n_splits())
print()
print("Best estimator pipeline structure:")
for name, step in grid_dt.best_estimator_.steps:
    print(f"  {name:12s} -> {step}")

MODEL B -- Decision Tree -- GridSearchCV results
Best parameters:        {'classifier__criterion': 'entropy', 'classifier__max_depth': 8, 'classifier__min_samples_leaf': 5, 'classifier__min_samples_split': 2}
Best CV score (average_precision): 0.2463
Number of CV folds:     5

Best estimator pipeline structure:
  preprocessing -> ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['loan_amnt', 'term', 'int_rate',
                                  'installment', 'annual_inc', 'dti',
                                  'inq_last_6mths', 'mths_since_last_delinq',
                                  'mths_since_last_record', 'open_acc',
                                  'revol_bal', 'revol_util', 'total_acc',
                                  'delinq_amnt', 'pub_rec_bankruptcies',
                                  'credit_hist

## 5. Leakage-safety & scope checks

Explicit confirmation before closing out Step 9.

In [ ]:
checks_passed = True

# 1. X_test / y_test are unchanged and were never scored/fit on.
check_1 = (X_test.shape == (8398, 34)) and (y_test.shape == (8398,))
print(f"[{'PASS' if check_1 else 'FAIL'}] X_test/y_test shapes unchanged from Step 7 "
      f"-> X_test={X_test.shape}, y_test={y_test.shape}")
checks_passed &= check_1

# 2. Both grids used the same 5-fold StratifiedKFold(shuffle=True, random_state=42).
check_2 = (cv.get_n_splits() == 5) and isinstance(cv, StratifiedKFold) and cv.shuffle
print(f"[{'PASS' if check_2 else 'FAIL'}] CV is StratifiedKFold(n_splits=5, shuffle=True) "
      f"-> n_splits={cv.get_n_splits()}, shuffle={cv.shuffle}")
checks_passed &= check_2

# 3. Both pipelines are imblearn Pipelines with SMOTE confined to fit-time.
check_3 = isinstance(pipeline_lr, ImbPipeline) and isinstance(pipeline_dt, ImbPipeline)
print(f"[{'PASS' if check_3 else 'FAIL'}] Both pipelines are imblearn.pipeline.Pipeline "
      f"-> {check_3}")
checks_passed &= check_3

# 4. Each pipeline used its own cloned preprocessor (Step 7's `preprocessor`
#    object, already fit on the full X_train above, was not reused inside
#    either grid search).
check_4 = (pipeline_lr.named_steps['preprocessing'] is not preprocessor
           and pipeline_dt.named_steps['preprocessing'] is not preprocessor)
print(f"[{'PASS' if check_4 else 'FAIL'}] Both pipelines use their own cloned "
      f"preprocessor -> {check_4}")
checks_passed &= check_4

# 5. Neither grid search's scorer or estimator ever saw X_test/y_test.
#    (By construction: only grid_lr.fit(X_train, y_train) and
#    grid_dt.fit(X_train, y_train) were called above.)
print(f"[PASS] X_test/y_test were never passed to .fit()/.score() in this notebook "
      f"-> confirmed by construction (see Sections 3-4)")

print()
print("=" * 60)
print(f"ALL CHECKS PASSED: {checks_passed}")
print("=" * 60)
assert checks_passed, "One or more Step 9 checks failed!"


[PASS] X_test/y_test shapes unchanged from Step 7 -> X_test=(8398, 34), y_test=(8398,)
[PASS] CV is StratifiedKFold(n_splits=5, shuffle=True) -> n_splits=5, shuffle=True
[PASS] Both pipelines are imblearn.pipeline.Pipeline -> True
[PASS] Both pipelines use their own cloned preprocessor -> True
[PASS] X_test/y_test were never passed to .fit()/.score() in this notebook -> confirmed by construction (see Sections 3-4)

ALL CHECKS PASSED: True


## Step 9 Summary

In [ ]:
summary_df = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'Pipeline': 'preprocessing -> StandardScaler -> PCA(0.95) -> SMOTE -> LogisticRegression',
        'Best Params': grid_lr.best_params_,
        f'Best CV {SCORING_METRIC}': round(grid_lr.best_score_, 4),
        'CV Folds': cv.get_n_splits(),
    },
    {
        'Model': 'Decision Tree',
        'Pipeline': 'preprocessing -> SMOTE -> DecisionTreeClassifier',
        'Best Params': grid_dt.best_params_,
        f'Best CV {SCORING_METRIC}': round(grid_dt.best_score_, 4),
        'CV Folds': cv.get_n_splits(),
    },
])

print("STEP 9 SUMMARY -- Model Training & Hyperparameter Tuning")
print("=" * 60)
for _, row in summary_df.iterrows():
    print(f"\n{row['Model']}")
    print(f"  Pipeline:      {row['Pipeline']}")
    print(f"  Best params:   {row['Best Params']}")
    print(f"  Best CV score: {row[f'Best CV {SCORING_METRIC}']} ({SCORING_METRIC})")
    print(f"  CV folds:      {row['CV Folds']}")

print()
print("Scoring metric used:            average_precision (PR-AUC) -- see Section 2")
print("Training data used:             original X_train/y_train (Step 8 diagnostic")
print("                                 fit_resample() arrays NOT used)")
print("SMOTE/PCA/scaling scope:        refit inside every CV training fold only")
print("X_test touched:                 NO")
print("Threshold tuning performed:     NO")
print("Random Forest / XGBoost added:  NO")
print("SHAP performed:                 NO")

summary_df

STEP 9 SUMMARY -- Model Training & Hyperparameter Tuning

Logistic Regression
  Pipeline:      preprocessing -> StandardScaler -> PCA(0.95) -> SMOTE -> LogisticRegression
  Best params:   {'classifier__C': 10, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}
  Best CV score: 0.2931 (average_precision)
  CV folds:      5

Decision Tree
  Pipeline:      preprocessing -> SMOTE -> DecisionTreeClassifier
  Best params:   {'classifier__criterion': 'entropy', 'classifier__max_depth': 8, 'classifier__min_samples_leaf': 5, 'classifier__min_samples_split': 2}
  Best CV score: 0.2463 (average_precision)
  CV folds:      5

Scoring metric used:            average_precision (PR-AUC) -- see Section 2
Training data used:             original X_train/y_train (Step 8 diagnostic
                                 fit_resample() arrays NOT used)
SMOTE/PCA/scaling scope:        refit inside every CV training fold only
X_test touched:                 NO
Threshold tuning performed:     NO
Rando

,Model,Pipeline,Best Params,Best CV average_precision,CV Folds
0,Logistic Regression,preprocessing -> StandardScaler -> PCA(0.95) -...,"{'classifier__C': 10, 'classifier__penalty': '...",0.2931,5
1,Decision Tree,preprocessing -> SMOTE -> DecisionTreeClassifier,"{'classifier__criterion': 'entropy', 'classifi...",0.2463,5


In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
pipeline_rf = ImbPipeline(steps=[
    ('preprocessing', clone(preprocessor)),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('classifier', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])

print("Pipeline C (Random Forest) structure:")
for name, step in pipeline_rf.steps:
    print(f"  {name:12s} -> {step}")

Pipeline C (Random Forest) structure:
  preprocessing -> ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['loan_amnt', 'term', 'int_rate',
                                  'installment', 'annual_inc', 'dti',
                                  'inq_last_6mths', 'mths_since_last_delinq',
                                  'mths_since_last_record', 'open_acc',
                                  'revol_bal', 'revol_util', 'total_acc',
                                  'delinq_amnt', 'pub_rec_bankruptcies',
                                  'credit_history_months', 'loan_to...
                                  'has_delinquency',
                                  'pub_rec_bankruptcies_missing',
                                  'revol_util_missing', 'emp_length_missing',
                                  'is_income_verified

In [ ]:
param_grid_rf = {
    'classifier__n_estimators': [200, 400],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 10],
    'classifier__min_samples_leaf': [1, 5],
    'classifier__max_features': ['sqrt'],
}

_n_combos_rf = (len(param_grid_rf['classifier__n_estimators'])
                * len(param_grid_rf['classifier__max_depth'])
                * len(param_grid_rf['classifier__min_samples_split'])
                * len(param_grid_rf['classifier__min_samples_leaf'])
                * len(param_grid_rf['classifier__max_features']))
print(f"Grid size: {_n_combos_rf} parameter combinations x {cv.get_n_splits()} folds "
      f"= {_n_combos_rf * cv.get_n_splits()} fits")

grid_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid_rf,
    scoring=SCORING_METRIC,
    cv=cv,
    n_jobs=-1,
    refit=True,
)

Grid size: 24 parameter combinations x 5 folds = 120 fits


In [ ]:
grid_rf.fit(X_train, y_train)  # X_test is never passed in here

/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median'))]),
                                                                         ['loan_amnt',
                                                                          'term',
                                                                          'int_rate',
                                                                          'installment',
                                                                          'annual_inc',
                                                                          'dti',
                                                                          'inq_last_6mths',
                                                                          'mths_since_last_delinq',
                                                                          'mths_since_last_record',
                                                                          '...
                                                                          'purpose_grouped'])])),
                                       ('smote', SMOTE(random_state=42)),
                                       ('classifier',
                                        RandomForestClassifier(n_jobs=-1,
                                                               random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__max_depth': [None, 10, 20],
                         'classifier__max_features': ['sqrt'],
                         'classifier__min_samples_leaf': [1, 5],
                         'classifier__min_samples_split': [2, 10],
                         'classifier__n_estimators': [200, 400]},
             scoring='average_precision')

In [ ]:
print("=" * 60)
print("MODEL C -- Random Forest -- GridSearchCV results")
print("=" * 60)
print("Best parameters:       ", grid_rf.best_params_)
print(f"Best CV score ({SCORING_METRIC}): {grid_rf.best_score_:.4f}")
print("Number of CV folds:    ", cv.get_n_splits())

MODEL C -- Random Forest -- GridSearchCV results
Best parameters:        {'classifier__max_depth': 20, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 10, 'classifier__n_estimators': 400}
Best CV score (average_precision): 0.2923
Number of CV folds:     5


In [ ]:
cv_comparison_df = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'Best CV PR-AUC': round(grid_lr.best_score_, 4),
        'Best Parameters': grid_lr.best_params_,
    },
    {
        'Model': 'Decision Tree',
        'Best CV PR-AUC': round(grid_dt.best_score_, 4),
        'Best Parameters': grid_dt.best_params_,
    },
    {
        'Model': 'Random Forest',
        'Best CV PR-AUC': round(grid_rf.best_score_, 4),
        'Best Parameters': grid_rf.best_params_,
    },
])

cv_comparison_df

,Model,Best CV PR-AUC,Best Parameters
0,Logistic Regression,0.2931,"{'classifier__C': 10, 'classifier__penalty': '..."
1,Decision Tree,0.2463,"{'classifier__criterion': 'entropy', 'classifi..."
2,Random Forest,0.2923,"{'classifier__max_depth': 20, 'classifier__max..."
